# Failure Analysis & Classification

**Purpose:** Identify and categorize failure modes in RL policies

**Analysis Includes:**
- Failure detection (error > threshold)
- Trajectory clustering (identify common failure patterns)
- Scenario-specific failure rates
- Root cause analysis
- Visualization of worst-case trajectories

**Goal:** Understand why and when the policy fails

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import json
from pathlib import Path
from mpl_toolkits.mplot3d import Axes3D

sns.set_style('whitegrid')
sns.set_context('notebook')

print("✅ Imports complete")

## 1. Load Test Results

In [ ]:
def load_trajectory_data(results_path):
    """
    Load full trajectory data from test results.
    
    Args:
        results_path: Path to JSON results
        
    Returns:
        Dictionary with trajectory data
    """
    with open(results_path, 'r') as f:
        data = json.load(f)
    
    trajectories = {}
    for scenario_key, scenario_data in data.items():
        trajectories[scenario_key] = {
            'errors': np.array(scenario_data.get('errors', [])),
            'positions': np.array(scenario_data.get('positions', [])),
            'targets': np.array(scenario_data.get('targets', [])),
        }
    
    return trajectories


# Load Phase 03 data
phase03_data = load_trajectory_data(
    '../../../visualization_tools/plotting/phase03_test_results.json'
)

print(f"✅ Loaded {len(phase03_data)} scenarios")
for key in phase03_data:
    n_traj = len(phase03_data[key]['errors'])
    print(f"  {key}: {n_traj} trajectories")

## 2. Failure Detection

In [ ]:
def classify_failures(errors, threshold=0.1):
    """
    Classify trajectories as success or failure.
    
    Args:
        errors: Array of final errors
        threshold: Success threshold in meters
        
    Returns:
        Dictionary with failure statistics
    """
    failures = errors > threshold
    
    return {
        'n_total': len(errors),
        'n_failures': np.sum(failures),
        'n_success': np.sum(~failures),
        'failure_rate': np.mean(failures) * 100,
        'success_rate': np.mean(~failures) * 100,
        'failure_indices': np.where(failures)[0],
        'worst_error': np.max(errors),
        'worst_index': np.argmax(errors),
    }


# Analyze failures per scenario
failure_analysis = {}

for scenario, data in phase03_data.items():
    analysis = classify_failures(data['errors'])
    failure_analysis[scenario] = analysis
    
    print(f"\n{scenario}:")
    print(f"  Success rate: {analysis['success_rate']:.1f}%")
    print(f"  Failures: {analysis['n_failures']}/{analysis['n_total']}")
    print(f"  Worst error: {analysis['worst_error']:.4f}m")

## 3. Failure Rate by Scenario

In [ ]:
# Create failure rate comparison plot
fig, ax = plt.subplots(figsize=(12, 6))

scenarios = list(failure_analysis.keys())
failure_rates = [failure_analysis[s]['failure_rate'] for s in scenarios]
success_rates = [failure_analysis[s]['success_rate'] for s in scenarios]

x = np.arange(len(scenarios))
width = 0.35

bars1 = ax.bar(x - width/2, success_rates, width, label='Success', color='#2ecc71', alpha=0.8)
bars2 = ax.bar(x + width/2, failure_rates, width, label='Failure', color='#e74c3c', alpha=0.8)

ax.set_ylabel('Percentage (%)', fontsize=12)
ax.set_title('Success vs Failure Rates by Scenario', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(scenarios, rotation=45, ha='right')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim([0, 105])

# Add value labels on bars
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        if height > 0:
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{height:.1f}%', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('failure_rate_by_scenario.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Failure rate plot saved")

## 4. Visualize Worst Failures

In [ ]:
def plot_worst_trajectories(scenario_data, scenario_name, n_worst=5):
    """
    Visualize the worst-performing trajectories.
    
    Args:
        scenario_data: Dictionary with errors and positions
        scenario_name: Name of scenario
        n_worst: Number of worst trajectories to show
    """
    errors = scenario_data['errors']
    positions = scenario_data['positions']
    targets = scenario_data['targets']
    
    if len(positions) == 0:
        print(f"⚠️ No position data for {scenario_name}")
        return
    
    # Get indices of worst trajectories
    worst_indices = np.argsort(errors)[-n_worst:][::-1]
    
    fig = plt.figure(figsize=(14, 10))
    
    # 3D plot
    ax1 = fig.add_subplot(2, 2, (1, 3), projection='3d')
    
    colors = plt.cm.Reds(np.linspace(0.5, 1, n_worst))
    
    for i, idx in enumerate(worst_indices):
        traj = positions[idx]
        target = targets[idx]
        error = errors[idx]
        
        # Plot trajectory
        ax1.plot(traj[:, 0], traj[:, 1], traj[:, 2], 
                color=colors[i], linewidth=2, alpha=0.7,
                label=f'Traj {idx} (err={error:.3f}m)')
        
        # Plot start and end
        ax1.scatter(*traj[0], color=colors[i], s=100, marker='o', 
                   edgecolors='black', linewidth=1.5, zorder=5)
        ax1.scatter(*traj[-1], color=colors[i], s=100, marker='X', 
                   edgecolors='black', linewidth=1.5, zorder=5)
        
        # Plot target
        ax1.scatter(*target[-1], color='green', s=200, marker='*', 
                   edgecolors='black', linewidth=2, zorder=10, alpha=0.8)
    
    ax1.set_xlabel('X (m)', fontsize=11)
    ax1.set_ylabel('Y (m)', fontsize=11)
    ax1.set_zlabel('Z (m)', fontsize=11)
    ax1.set_title(f'Worst {n_worst} Trajectories: {scenario_name}', 
                 fontsize=13, fontweight='bold')
    ax1.legend(fontsize=9, loc='upper left')
    ax1.grid(True, alpha=0.3)
    
    # Error distribution
    ax2 = fig.add_subplot(2, 2, 2)
    ax2.hist(errors, bins=30, color='steelblue', alpha=0.7, edgecolor='black')
    
    # Mark worst trajectories
    for idx in worst_indices:
        ax2.axvline(errors[idx], color='red', linestyle='--', linewidth=1.5, alpha=0.7)
    
    ax2.set_xlabel('Final Error (m)', fontsize=11)
    ax2.set_ylabel('Frequency', fontsize=11)
    ax2.set_title('Error Distribution (red = worst cases)', fontsize=12)
    ax2.grid(True, alpha=0.3)
    
    # Box plot
    ax3 = fig.add_subplot(2, 2, 4)
    bp = ax3.boxplot([errors], vert=False, patch_artist=True, widths=0.6)
    bp['boxes'][0].set_facecolor('steelblue')
    bp['boxes'][0].set_alpha(0.7)
    
    # Mark worst cases
    for idx in worst_indices:
        ax3.scatter(errors[idx], 1, color='red', s=100, marker='X', 
                   edgecolors='black', linewidth=1.5, zorder=5)
    
    ax3.set_xlabel('Final Error (m)', fontsize=11)
    ax3.set_title('Error Box Plot', fontsize=12)
    ax3.grid(True, alpha=0.3)
    ax3.set_yticks([])
    
    plt.tight_layout()
    plt.savefig(f'worst_failures_{scenario_name}.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"✅ Worst trajectories plot saved for {scenario_name}")


# Plot worst trajectories for high-intensity scenarios
if 'continuous_high' in phase03_data:
    plot_worst_trajectories(
        phase03_data['continuous_high'],
        'Continuous High',
        n_worst=3
    )
else:
    print("⚠️ No trajectory data available for visualization")

## 5. Failure Pattern Clustering

In [ ]:
def cluster_failures(positions, errors, threshold=0.1, n_clusters=3):
    """
    Cluster failure trajectories to identify common patterns.
    
    Args:
        positions: Array of trajectory positions
        errors: Array of final errors
        threshold: Failure threshold
        n_clusters: Number of clusters
        
    Returns:
        Cluster labels and centroids
    """
    if len(positions) == 0:
        return None, None
    
    # Select only failures
    failure_mask = errors > threshold
    if np.sum(failure_mask) < n_clusters:
        print(f"⚠️ Not enough failures for clustering ({np.sum(failure_mask)} < {n_clusters})")
        return None, None
    
    failure_positions = positions[failure_mask]
    
    # Extract features: final position and trajectory spread
    features = []
    for traj in failure_positions:
        final_pos = traj[-1]
        traj_std = np.std(traj, axis=0)
        features.append(np.concatenate([final_pos, traj_std]))
    
    features = np.array(features)
    
    # Standardize features
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(features)
    
    # K-means clustering
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    labels = kmeans.fit_predict(features_scaled)
    
    return labels, kmeans.cluster_centers_


# Cluster failures for a scenario
if 'impulse_high' in phase03_data:
    data = phase03_data['impulse_high']
    if len(data['positions']) > 0:
        labels, centroids = cluster_failures(
            data['positions'],
            data['errors'],
            threshold=0.1,
            n_clusters=3
        )
        
        if labels is not None:
            print("✅ Failure clustering complete")
            print(f"Cluster distribution:")
            unique, counts = np.unique(labels, return_counts=True)
            for cluster, count in zip(unique, counts):
                print(f"  Cluster {cluster}: {count} failures")
    else:
        print("⚠️ No position data for clustering")
else:
    print("⚠️ No data for 'impulse_high' scenario")

## 6. Root Cause Analysis

In [ ]:
def analyze_failure_causes(scenario_data, analysis):
    """
    Investigate potential root causes of failures.
    
    Possible causes:
    - Large initial distance
    - Complex trajectory requirements
    - Insufficient trajectory length
    - Poor policy convergence
    """
    errors = scenario_data['errors']
    positions = scenario_data['positions']
    targets = scenario_data['targets']
    
    if len(positions) == 0:
        return
    
    failure_indices = analysis['failure_indices']
    
    if len(failure_indices) == 0:
        print("✅ No failures to analyze!")
        return
    
    # Compute characteristics for failures vs successes
    success_mask = ~np.isin(np.arange(len(errors)), failure_indices)
    
    # Initial distances
    initial_distances = []
    for i in range(len(positions)):
        dist = np.linalg.norm(positions[i][0] - targets[i][-1])
        initial_distances.append(dist)
    
    initial_distances = np.array(initial_distances)
    
    # Trajectory lengths
    traj_lengths = np.array([len(traj) for traj in positions])
    
    # Trajectory complexity (total path length)
    path_lengths = []
    for traj in positions:
        path_len = np.sum(np.linalg.norm(np.diff(traj, axis=0), axis=1))
        path_lengths.append(path_len)
    path_lengths = np.array(path_lengths)
    
    print("\n📊 Root Cause Analysis:")
    print("\nFailures vs Successes:")
    print(f"  Initial distance:")
    print(f"    Failures: {np.mean(initial_distances[failure_indices]):.4f} ± {np.std(initial_distances[failure_indices]):.4f}m")
    print(f"    Successes: {np.mean(initial_distances[success_mask]):.4f} ± {np.std(initial_distances[success_mask]):.4f}m")
    
    print(f"  Trajectory length:")
    print(f"    Failures: {np.mean(traj_lengths[failure_indices]):.1f} ± {np.std(traj_lengths[failure_indices]):.1f} steps")
    print(f"    Successes: {np.mean(traj_lengths[success_mask]):.1f} ± {np.std(traj_lengths[success_mask]):.1f} steps")
    
    print(f"  Path complexity:")
    print(f"    Failures: {np.mean(path_lengths[failure_indices]):.4f} ± {np.std(path_lengths[failure_indices]):.4f}m")
    print(f"    Successes: {np.mean(path_lengths[success_mask]):.4f} ± {np.std(path_lengths[success_mask]):.4f}m")


# Analyze failures for a scenario
if 'continuous_high' in phase03_data and 'continuous_high' in failure_analysis:
    analyze_failure_causes(
        phase03_data['continuous_high'],
        failure_analysis['continuous_high']
    )
else:
    print("⚠️ No data for root cause analysis")

## 7. Summary Report

In [ ]:
def generate_failure_report(failure_analysis):
    """
    Generate comprehensive failure analysis report.
    """
    report = []
    report.append("=" * 60)
    report.append("FAILURE ANALYSIS REPORT")
    report.append("=" * 60)
    
    total_tests = sum(a['n_total'] for a in failure_analysis.values())
    total_failures = sum(a['n_failures'] for a in failure_analysis.values())
    overall_success_rate = (1 - total_failures / total_tests) * 100
    
    report.append(f"\nOverall Statistics:")
    report.append(f"  Total tests: {total_tests}")
    report.append(f"  Total failures: {total_failures}")
    report.append(f"  Overall success rate: {overall_success_rate:.2f}%")
    
    report.append(f"\nPer-Scenario Breakdown:")
    for scenario, analysis in failure_analysis.items():
        report.append(f"\n  {scenario}:")
        report.append(f"    Success rate: {analysis['success_rate']:.1f}%")
        report.append(f"    Failures: {analysis['n_failures']}/{analysis['n_total']}")
        report.append(f"    Worst error: {analysis['worst_error']:.4f}m")
    
    # Identify most challenging scenario
    worst_scenario = max(failure_analysis.items(), key=lambda x: x[1]['failure_rate'])
    report.append(f"\n⚠️ Most Challenging Scenario: {worst_scenario[0]}")
    report.append(f"   Failure rate: {worst_scenario[1]['failure_rate']:.1f}%")
    
    # Identify best scenario
    best_scenario = min(failure_analysis.items(), key=lambda x: x[1]['failure_rate'])
    report.append(f"\n✅ Best Scenario: {best_scenario[0]}")
    report.append(f"   Success rate: {best_scenario[1]['success_rate']:.1f}%")
    
    report.append("\n" + "=" * 60)
    
    return "\n".join(report)


# Generate report
if failure_analysis:
    report = generate_failure_report(failure_analysis)
    print(report)
    
    # Save to file
    with open('failure_analysis_report.txt', 'w') as f:
        f.write(report)
    
    print("\n✅ Report saved to: failure_analysis_report.txt")
else:
    print("⚠️ No failure data to report")

## Summary

**This notebook provides:**

✅ Failure detection and classification  
✅ Per-scenario failure rate analysis  
✅ Visualization of worst-case trajectories  
✅ Failure pattern clustering  
✅ Root cause investigation  
✅ Comprehensive failure reports  

**Key Insights:**
- Identify which scenarios are most challenging
- Understand common failure patterns
- Guide improvements for Phase 04-05

**Next Steps:**
1. Use insights to design targeted improvements
2. Compare failure rates: Phase 03 vs Phase 04
3. Track failure reduction over training phases